# Building RAG Pipelines — C8-W4-S1
## Notebook 03: Chunking — Slicing Documents
**Duration:** 20 min &nbsp;|&nbsp; **Mode:** Conceptual + Guided Coding &nbsp;|&nbsp; upGrad Live Session

> Taught **WHY → WHAT → HOW**. We keep asking *"What happens if this step is poorly
> designed?"* and we **predict before we run** and **compare outputs**. LangChain is
> shown as a **parallel mapping** — it abstracts mechanics but not design decisions.

![pipeline](https://dummyimage.com/1000x70/1f2937/ffffff&text=Loading+%E2%86%92+Chunking+%E2%86%92+Retrieval+%E2%86%92+Augmentation+%E2%86%92+Generation+%E2%86%92+Evaluation)

In [ ]:
# ============================================================
# COLAB BOOTSTRAP — run this cell first. (Same as every notebook.)
# ============================================================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/building-rag-pipelines.git"  # INSTRUCTOR: set this

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

_pip("numpy", "openai", "tiktoken", "rank-bm25", "beautifulsoup4", "pypdf")
try:
    import rag_pipeline
except ModuleNotFoundError:
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
        if os.path.isdir("building-rag-pipelines"):
            sys.path.insert(0, "building-rag-pipelines")
        else:
            print("Clone failed. Upload `rag_pipeline/` + `data/` via the Colab file browser, then re-run.")
    else:
        sys.path.insert(0, os.path.abspath(".."))
    import rag_pipeline

def data_path(*parts):
    for base in ("data", "../data", "building-rag-pipelines/data"):
        p = os.path.join(base, *parts)
        if os.path.exists(p):
            return p
    return os.path.join("data", *parts)

print("rag_pipeline", rag_pipeline.__version__, "ready.  Colab:", IN_COLAB)

In [ ]:
# Providers: OpenAI by default; offline MOCK if no key (class always runs).
import os
if not os.getenv("OPENAI_API_KEY"):
    os.environ["RAG_LLM_PROVIDER"] = "mock"
    os.environ["RAG_EMBED_PROVIDER"] = "mock"
# For the real stack: set OPENAI_API_KEY (getpass or Colab userdata) BEFORE this cell.
from rag_pipeline import config
print(config.current_config())

In [ ]:
from rag_pipeline.loaders import load_directory
docs = load_directory(data_path("corpus"))
print(f"Loaded {len(docs)} documents from the Acme Cloud corpus.")

## WHY — chunking is where retrieval quality is decided

Two hard limits force chunking:
1. **Embeddings average meaning.** A whole document compresses into ONE vector;
   the longer the passage, the more the relevant sentence is drowned out.
2. **Context & cost are finite.** You can't stuff whole documents into the prompt.

This is the session's central cause→effect chain:

> **bad chunking → poor retrieval → bad answer.**

- Chunks **too LARGE** → the embedding is diluted; the query's target sentence is
  averaged away with unrelated text, similarity drops, you **miss** it.
- Chunks **too SMALL** → ideas fragment across boundaries; the answer needs two
  chunks that never co-occur, so the model gets **half the story**.
- Splitting **mid-sentence / mid-table** destroys meaning outright.

## WHAT — three strategies + the overlap knob

| Strategy | How it splits | Trade-off |
|----------|---------------|-----------|
| **Fixed-size** | every N chars/tokens | simple & fast, but **structure-blind** (cuts mid-sentence) |
| **Recursive** | largest natural boundary that fits (¶ → line → sentence → word) | structure-aware; the **sensible default** |
| **Semantic** | where the topic shifts (embedding-similarity drop) | best boundaries, **highest cost** (embed every sentence) |

**Overlap** repeats a little text between neighbours so an idea straddling a
boundary still appears whole in at least one chunk.

**DO** preserve semantic boundaries; add a modest 10–20% overlap.
**DON'T** make chunks so big signal is diluted, or so small ideas fragment.

## HOW (from scratch) — see the mechanics

In [ ]:
from rag_pipeline.chunking import (
    fixed_size_chunk, recursive_chunk, semantic_chunk, chunk_stats
)
from rag_pipeline import config
emb = config.get_embedder()

sample = [d for d in docs if "pricing" in d.metadata["source"]]

fixed  = fixed_size_chunk(sample, chunk_size=300, overlap=50)
recur  = recursive_chunk(sample, chunk_size=300, overlap=50)

print("FIXED    ", chunk_stats(fixed))
print("RECURSIVE", chunk_stats(recur))
print("\n--- FIXED chunk boundary (watch it cut mid-sentence) ---")
print(repr(fixed[0].page_content[-80:]), "|", repr(fixed[1].page_content[:80]))
print("\n--- RECURSIVE chunk boundary (breaks on paragraphs) ---")
print(repr(recur[0].page_content[-80:]))

> ### ✋ Predict before you run
> We'll re-chunk the SAME pricing document at chunk_size ∈ {150, 400, 1000} and then ask 'How much does the Growth plan cost?'. Which size will retrieve the cleanest, most on-target chunk? Why might 150 fail? Why might 1000 fail?
>
> *Commit to a guess before executing. Comparing prediction vs result is the point.*

In [ ]:
# Compare across chunk sizes — the agenda's 'compare outputs across chunk sizes'.
from rag_pipeline.vectorstore import InMemoryVectorStore
from rag_pipeline.retrieval import DenseRetriever
from rag_pipeline.embeddings import embed_documents

question = "How much does the Growth plan cost per month?"

for size in (150, 400, 1000):
    chunks = recursive_chunk(sample, chunk_size=size, overlap=int(size*0.15))
    store = InMemoryVectorStore().add(chunks, embed_documents(emb, chunks))
    top = DenseRetriever(store, emb).retrieve(question, k=1)[0]
    doc, score = top
    print(f"size={size:>4}  n_chunks={len(chunks):>2}  score={score:.3f}")
    print(f"           top chunk -> {doc.page_content[:110].strip()!r}")
    print()

**What you should observe:** very small chunks fragment the pricing table (the
'$199' can land in a different chunk from 'Growth'), while very large chunks
dilute the embedding (the Growth line is averaged in with Starter/Enterprise).
A middle size with overlap usually wins. **This is design decision #1 that the
LLM can never fix for you.**

## HOW — semantic chunking (topic-aware boundaries)

Semantic chunking embeds each sentence and cuts where consecutive sentences are
*least* similar — i.e. where the topic changes. Best coherence, but one embedding
call per sentence.

In [ ]:
sem = semantic_chunk(sample, emb, threshold_percentile=85)
print("SEMANTIC", chunk_stats(sem))
for i, c in enumerate(sem[:3]):
    print(f"\n[chunk {i}] {c.page_content[:160].strip()}...")

## HOW (parallel mapping) — LangChain TextSplitters

`RecursiveCharacterTextSplitter` is the framework version of our `recursive_chunk`
— same separator ladder, same size/overlap knobs. The framework abstracts the
loop; **you still choose** size, overlap, and what counts as a boundary.

In [ ]:
try:
    from rag_pipeline.chunking import recursive_chunk_langchain
    lc_chunks = recursive_chunk_langchain(sample, chunk_size=300, overlap=50)
    print(f"LangChain produced {len(lc_chunks)} chunks (mapped back to our Document type).")
    print(chunk_stats(lc_chunks))
except Exception as e:
    print("LangChain not installed — concept still holds:", e)

## Recap
- Chunking is the first place retrieval quality is won or lost.
- Fixed = simple/blind; recursive = structure-aware default; semantic = best/costliest.
- **Tune size + overlap by measuring**, not by folklore. Predict, run, compare.

**Next → Notebook 04 (Retrieval):** turn chunks into vectors and *find* the right
ones — dense, sparse (BM25), hybrid, filtering, reranking.